In [10]:
import subprocess
import threading
import time
import socket
import os

# Assuming WORKSPACE is defined in a previous cell, e.g., /content/drive/MyDrive/ComfyUI
# If not, you might need to re-evaluate the cell where WORKSPACE is set (e.g., cell bbbbbbbbbb)
%cd /content/drive/MyDrive/ComfyUI

# Install missing dependencies as suggested by the error messages
print("\n--- Installing ComfyUI requirements ---\n")
!pip install -r requirements.txt
# Explicitly install alembic and blake3, as they were mentioned in errors/warnings
!pip install alembic blake3

# List custom nodes to diagnose potential issues like 'comfy_aimdo'
print("\n--- Listing Custom Nodes ---")
!ls -R custom_nodes

# Ensure custom node dependencies are restored right before starting ComfyUI
print("\n--- Restoring Custom Node Dependencies via ComfyUI-Manager ---\n")
!python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies

# Event to signal when ComfyUI is ready
comfyui_ready_event = threading.Event()

# List to keep track of all launched subprocesses for cleanup
running_subprocesses = []

def monitor_cloudflared_output(process):
    """Continuously reads and prints cloudflared's output, looking for the public URL."""
    url_found = False
    # Read both stderr and stdout
    for line in iter(process.stderr.readline, b''):
        l = line.decode().strip()
        print(f"[cloudflared stderr] {l}")
        if not url_found and ".trycloudflare.com" in l:
            try:
                # Attempt to extract the URL, assuming it's a single URL on the line
                # or within a larger string. This simple check might be enough.
                parts = l.split()
                for part in parts:
                    if part.startswith("https://") and ".trycloudflare.com" in part:
                        print("This is the URL to access ComfyUI:", part)
                        url_found = True
                        break
            except Exception as e:
                print(f"[cloudflared error] Error parsing URL from line: {l} - {e}")

    for line in iter(process.stdout.readline, b''):
        l = line.decode().strip()
        print(f"[cloudflared stdout] {l}")
        if not url_found and ".trycloudflare.com" in l:
            try:
                parts = l.split()
                for part in parts:
                    if part.startswith("https://") and ".trycloudflare.com" in part:
                        print("This is the URL to access ComfyUI:", part)
                        url_found = True
                        break
            except Exception as e:
                print(f"[cloudflared error] Error parsing URL from line: {l} - {e}")


def monitor_comfyui_output(process, ready_event):
    """Monitors ComfyUI's stdout/stderr and sets an event when it's ready."""
    try:
        # Prioritize stdout for the ready message
        for line in iter(process.stdout.readline, b''):
            l = line.decode().strip()
            print(f"[ComfyUI stdout] {l}")
            if "Starting server on" in l or "Uvicorn running on" in l or "To see the GUI go to:" in l:
                ready_event.set()
                # Removed time.sleep(2) to avoid potential race conditions where ComfyUI crashes during sleep
                # The socket check in iframe_thread will handle stabilization.
            # We check poll() at the end of each readline iteration to catch immediate crashes.
            if process.poll() is not None and ready_event.is_set():
                print(f"[ComfyUI monitor] ComfyUI process terminated with code {process.returncode} shortly after signalling readiness.")
                return
            elif process.poll() is not None:
                print(f"[ComfyUI monitor] ComfyUI process terminated with code {process.returncode} before signalling readiness.")
                return

        for line in iter(process.stderr.readline, b''):
            l = line.decode().strip()
            print(f"[ComfyUI stderr] {l}")
            if "Starting server on" in l or "Uvicorn running on" in l or "To see the GUI go to:" in l:
                ready_event.set()
                # Removed time.sleep(2)
            if process.poll() is not None and ready_event.is_set():
                print(f"[ComfyUI monitor] ComfyUI process terminated with code {process.returncode} shortly after signalling readiness.")
                return
            elif process.poll() is not None:
                print(f"[ComfyUI monitor] ComfyUI process terminated with code {process.returncode} before signalling readiness.")
                return

    except Exception as e:
        print(f"[ComfyUI monitor error] Error monitoring ComfyUI output: {e}")


def iframe_thread(port, ready_event):
  print("Waiting for ComfyUI server to start and become stable...")
  # Use a longer timeout for waiting for the event, as the ComfyUI startup can be slow.
  if not ready_event.wait(timeout=480): # Increased timeout to 8 minutes
      print("[ERROR] ComfyUI server did not signal readiness within the expected time. Cloudflared will not be launched.")
      return

  # Double-check if the port is actually open after ComfyUI signals readiness
  for _ in range(20): # Increased retry attempts
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      sock.close()
      if result == 0:
          print(f"ComfyUI confirmed listening on port {port}.")
          break
      time.sleep(1) # Wait a bit before retrying
  else:
      print(f"[ERROR] ComfyUI port {port} is not reachable after signaling readiness. Cloudflared will not be launched.")
      return

  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  # Launch cloudflared tunnel with the 'run' subcommand
  cloudflared_process = subprocess.Popen([
      "cloudflared", "tunnel", "run",
      "--token", "eyJhIjoiOTYzY2I4OTYwNzJmYTRkY2IyZWEyMTMzYWM4NDQ1MmEiLCJ0IjoiOTE2ZGQxMTctNzgxZC00OTVjLTlkNDctMjE4NWZkMTQ4ZWUzIiwicyI6Ik1XTTROR0UzT0dJdFpUWTFZeTAwTWpNNExUazRZVFl0T0RjMFpXWmpPVFkyTXpZMyJ9",
      "--url", f"http://127.0.0.1:{port}"
  ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

  running_subprocesses.append(cloudflared_process)

  # Start a separate daemon thread to monitor cloudflared's output
  threading.Thread(target=monitor_cloudflared_output, args=(cloudflared_process,), daemon=True).start()
  # The iframe_thread now finishes, letting cloudflared_process run in background, monitored by its own thread.


print("\n--- Starting ComfyUI server ---\n")

# Start ComfyUI in a non-blocking way using subprocess.Popen
# Removed --dont-print-server to get full output, including "Starting server on..."
comfyui_process = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0"],
    cwd=os.getcwd(), # Ensure it runs in the ComfyUI directory
    stdout=subprocess.PIPE, # Capture stdout
    stderr=subprocess.PIPE # Capture stderr
)

running_subprocesses.append(comfyui_process)

# Start monitoring ComfyUI output in a separate daemon thread
threading.Thread(target=monitor_comfyui_output, args=(comfyui_process, comfyui_ready_event,), daemon=True).start()

# Start the iframe_thread in the background, passing the event
threading.Thread(target=iframe_thread, daemon=True, args=(8188, comfyui_ready_event,)).start()

# Add a short delay to allow the monitoring threads to start and capture initial output
time.sleep(2)

# Keep the main thread alive indefinitely to support the daemon threads and subprocesses.
# This loop will run until the cell is manually interrupted or ComfyUI terminates unexpectedly.
try:
    while True:
        time.sleep(1) # Keep the main thread active
        if comfyui_process.poll() is not None:
            print(f"[CRITICAL ERROR] ComfyUI process terminated unexpectedly with return code: {comfyui_process.returncode}")
            comfyui_ready_event.set() # Unblock iframe_thread if still waiting
            # If ComfyUI dies, the tunnel is useless. We should exit.
            break
except KeyboardInterrupt:
    print("\nStopping ComfyUI and Cloudflared...")
finally:
    # Ensure all spawned subprocesses are terminated on cell stop
    print("\n--- Cleaning up processes ---")
    for p in running_subprocesses:
        if p.poll() is None: # If process is still running
            print(f"Terminating process: {p.args}")
            p.terminate()
            try:
                p.wait(timeout=5) # Wait for process to terminate gracefully
            except subprocess.TimeoutExpired:
                print(f"Process {p.args} did not terminate gracefully, killing it...")
                p.kill()
                p.wait()
        else:
            print(f"Process already terminated: {p.args} (Return code: {p.returncode})")
    print("Cleanup complete.")

/content/drive/MyDrive/ComfyUI

--- Installing ComfyUI requirements ---


--- Listing Custom Nodes ---
custom_nodes:
ComfyUI-Manager  example_node.py.example  __pycache__  websocket_image_save.py

custom_nodes/ComfyUI-Manager:
alter-list.json		 github-stats.json	      prestartup_script.py
CHANGELOG.md		 glob			      __pycache__
channels.list.template	 __init__.py		      pyproject.toml
check.bat		 js			      README.md
check.sh		 json-checker.py	      requirements.txt
cm-cli.py		 LICENSE.txt		      ruff.toml
cm-cli.sh		 misc			      scanner.py
components		 model-list.json	      scan.sh
custom-node-list.json	 node_db		      scripts
docs			 notebooks		      snapshots
extension-node-map.json  openapi.yaml		      tests
extras.json		 pip_overrides.json.template
git_helper.py		 pip_overrides.osx.template

custom_nodes/ComfyUI-Manager/components:

custom_nodes/ComfyUI-Manager/docs:
en  ko	README.md

custom_nodes/ComfyUI-Manager/docs/en:
cm-cli.md  use_aria2.md  v3.38-userdata-security-migration

### Inspecting `comfy_aimdo` Installation

In [12]:
# Attempt to import comfy_aimdo and print its file path
try:
    import comfy_aimdo
    print(f"Found comfy_aimdo installed at: {comfy_aimdo.__file__}")

    # List contents of the installed package directory
    package_dir = os.path.dirname(comfy_aimdo.__file__)
    print(f"Contents of {package_dir}:")
    !ls -l {package_dir}

    # Check specifically for a 'control' module or directory
    if os.path.exists(os.path.join(package_dir, 'control.py')):
        print("control.py found inside comfy_aimdo package.")
    elif os.path.exists(os.path.join(package_dir, 'control')):
        print("control directory found inside comfy_aimdo package. Listing contents:")
        !ls -l {os.path.join(package_dir, 'control')}
    else:
        print("Neither control.py nor control directory found inside comfy_aimdo package.")

except ImportError:
    print("comfy_aimdo module not found despite pip reporting it as satisfied. This is unexpected.")
except Exception as e:
    print(f"An error occurred: {e}")


Found comfy_aimdo installed at: None
An error occurred: expected str, bytes or os.PathLike object, not NoneType


Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [1]:
# #@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}
OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /

    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f ".ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/nightly/windows_base_files/run_nvidia_gpu.bat" ] && chmod 755 .ci/nightly/windows_base_files/run_nvidia_gpu.bat
  ![ -f ".ci/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows/update.py" ] && chmod 755 .ci/update_windows/update.py
  ![ -f ".ci/update_windows/update_comfyui.bat" ] && chmod 755 .ci/update_windows/update_comfyui.bat
  ![ -f ".ci/update_windows/README_VERY_IMPORTANT.txt" ] && chmod 755 .ci/update_windows/README_VERY_IMPORTANT.txt
  ![ -f ".ci/update_windows/run_cpu.bat" ] && chmod 755 .ci/update_windows/run_cpu.bat
  ![ -f ".ci/update_windows/run_nvidia_gpu.bat" ] && chmod 755 .ci/update_windows/run_nvidia_gpu.bat

  !git pull

!echo -= Install dependencies =-
!pip3 install accelerate
!pip3 install einops transformers>=4.28.1 safetensors>=0.4.2 aiohttp pyyaml Pillow scipy tqdm psutil tokenizers>=0.13.3
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip3 install torchsde
!pip3 install kornia>=0.7.1 spandrel soundfile sentencepiece

if OPTIONS['USE_COMFYUI_MANAGER']:
  %cd custom_nodes

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
  ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
  ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
  ![ -f "ComfyUI-Manager/node_db/tutorial/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/tutorial/scan.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat

  ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
  %cd ComfyUI-Manager
  !git pull

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
  !echo -= Install custom nodes dependencies =-
  !pip install GitPython
  !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies


Mounting Google Drive...
/
Mounted at /content/drive
/content/drive/MyDrive
-= Initial setup ComfyUI =-
Cloning into 'ComfyUI'...
remote: Enumerating objects: 48887, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 48887 (delta 3), reused 0 (delta 0), pack-reused 48880 (from 2)
Receiving objects: 100% (48887/48887), 88.63 MiB | 10.57 MiB/s, done.
Resolving deltas: 100% (33250/33250), done.
Updating files: 100% (1084/1084), done.
/content/drive/MyDrive/ComfyUI
-= Updating ComfyUI =-
Already up to date.
-= Install dependencies =-
Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 3.4 MB/s eta 0:00:00
/content/drive/MyDrive/ComfyUI/custom_nodes
-= Initial setup ComfyUI-Manager =-
Cloning into 'ComfyUI-Manager'...
remote: Enumerating objects: 32252, done.
remote: Counting objects: 100% (768/768), done.
remote: Compressing objects: 100% (366/366), done.
remote: Tota

Download some models/checkpoints/vae or custom comfyui nodes (uncomment the commands for the ones you want)

In [2]:
# Checkpoints

### SDXL
### I recommend these workflow examples: https://comfyanonymous.github.io/ComfyUI_examples/sdxl/

#!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-refiner-1.0/resolve/main/sd_xl_refiner_1.0.safetensors -P ./models/checkpoints/

# SDXL ReVision
#!wget -c https://huggingface.co/comfyanonymous/clip_vision_g/resolve/main/clip_vision_g.safetensors -P ./models/clip_vision/

# SD1.5
!wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P ./models/checkpoints/

# SD2
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-2-1-base/resolve/main/v2-1_512-ema-pruned.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-2-1/resolve/main/v2-1_768-ema-pruned.safetensors -P ./models/checkpoints/

# Some SD1.5 anime style
#!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A1_orangemixs.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A3_orangemixs.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/Linaqruf/anything-v3.0/resolve/main/anything-v3-fp16-pruned.safetensors -P ./models/checkpoints/

# Waifu Diffusion 1.5 (anime style SD2.x 768-v)
#!wget -c https://huggingface.co/waifu-diffusion/wd-1-5-beta3/resolve/main/wd-illusion-fp16.safetensors -P ./models/checkpoints/


# unCLIP models
#!wget -c https://huggingface.co/comfyanonymous/illuminatiDiffusionV1_v11_unCLIP/resolve/main/illuminatiDiffusionV1_v11-unclip-h-fp16.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/comfyanonymous/wd-1.5-beta2_unCLIP/resolve/main/wd-1-5-beta2-aesthetic-unclip-h-fp16.safetensors -P ./models/checkpoints/


# VAE
!wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/
#!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/VAEs/orangemix.vae.pt -P ./models/vae/
#!wget -c https://huggingface.co/hakurei/waifu-diffusion-v1-4/resolve/main/vae/kl-f8-anime2.ckpt -P ./models/vae/


# Loras
#!wget -c https://civitai.com/api/download/models/10350 -O ./models/loras/theovercomer8sContrastFix_sd21768.safetensors #theovercomer8sContrastFix SD2.x 768-v
#!wget -c https://civitai.com/api/download/models/10638 -O ./models/loras/theovercomer8sContrastFix_sd15.safetensors #theovercomer8sContrastFix SD1.x
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_offset_example-lora_1.0.safetensors -P ./models/loras/ #SDXL offset noise lora


# T2I-Adapter
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_depth_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_seg_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_sketch_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_keypose_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_openpose_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_color_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_canny_sd14v1.pth -P ./models/controlnet/

# T2I Styles Model
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_style_sd14v1.pth -P ./models/style_models/

# CLIPVision model (needed for styles model)
#!wget -c https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/pytorch_model.bin -O ./models/clip_vision/clip_vit14.bin


# ControlNet
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11e_sd15_ip2p_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11e_sd15_shuffle_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_canny_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11f1p_sd15_depth_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_inpaint_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_lineart_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_mlsd_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_normalbae_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_openpose_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_scribble_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_seg_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_softedge_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15s2_lineart_anime_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11u_sd15_tile_fp16.safetensors -P ./models/controlnet/

# ControlNet SDXL
#!wget -c https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-canny-rank256.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-depth-rank256.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-recolor-rank256.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-sketch-rank256.safetensors -P ./models/controlnet/

# Controlnet Preprocessor nodes by Fannovel16
#!cd custom_nodes && git clone https://github.com/Fannovel16/comfy_controlnet_preprocessors; cd comfy_controlnet_preprocessors && python install.py


# GLIGEN
#!wget -c https://huggingface.co/comfyanonymous/GLIGEN_pruned_safetensors/resolve/main/gligen_sd14_textbox_pruned_fp16.safetensors -P ./models/gligen/


# ESRGAN upscale model
#!wget -c https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P ./models/upscale_models/
#!wget -c https://huggingface.co/sberbank-ai/Real-ESRGAN/resolve/main/RealESRGAN_x2.pth -P ./models/upscale_models/
#!wget -c https://huggingface.co/sberbank-ai/Real-ESRGAN/resolve/main/RealESRGAN_x4.pth -P ./models/upscale_models/




--2026-09-05 17:29:37--  https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.23, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.118|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt [following]
--2026-09-05 17:29:37--  https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66d19580e2632490a6bc5829/773d9e5ddbf6a84612066b3ea00a0bcfe4ba870a9befc88816f71b5927e76e38?user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27v1-5-pruned-emaonly.ckpt%3B+filename%3D%22v1-5-pruned-emaonly.ckpt%22%3B&X-Xet-Cas-Uid=public

### Run ComfyUI with cloudflared (Recommended Way)




In [7]:
!wget -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server
!cloudflared tunnel run --token eyJhIjoiOTYzY2I4OTYwNzJmYTRkY2IyZWEyMTMzYWM4NDQ1MmEiLCJ0IjoiOTE2ZGQxMTctNzgxZC00OTVjLTlkNDctMjE4NWZkMTQ4ZWUzIiwicyI6Ik1XTTROR0UzT0dJdFpUWTFZeTAwTWpNNExUazRZVFl0T0RjMFpXWmpPVFkyTXpZMyJ9


--2026-09-05 17:37:54--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.8.3/cloudflared-linux-amd64.deb [following]
--2026-09-05 17:37:54--  https://github.com/cloudflare/cloudflared/releases/download/2026.8.3/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/b86e637a-98e9-4791-9df9-f9674a251590?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-09-05T18%3A19%3A50Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

In [4]:
import subprocess
import threading
import time
import socket
import os

# Assuming WORKSPACE is defined in a previous cell, e.g., /content/drive/MyDrive/ComfyUI
# If not, you might need to re-evaluate the cell where WORKSPACE is set (e.g., cell bbbbbbbbbb)
%cd /content/drive/MyDrive/ComfyUI

# Install missing dependencies as suggested by the error messages
print("\n--- Installing ComfyUI requirements ---\n")
!pip install -r requirements.txt

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  # Launch cloudflared tunnel
  # The token is passed directly in the command
  p = subprocess.Popen(["cloudflared", "tunnel", "--token", "eyJhIjoiOTYzY2I4OTYwNzJmYTRkY2IyZWEyMTMzYWM4NDQ1MmEiLCJ0IjoiOTE2ZGQxMTctNzgxZC00OTVjLTlkNDctMjE4NWZkMTQ4ZWUzIiwicyI6Ik1XTTROR0UzT0dJdFpUWTFZeTAwTWpNNExUazRZVFl0T0RjMFpXWmpPVFkyTXpZMyJ9",  "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

  # Print all output from cloudflared to diagnose where the URL is printed.
  # This will help us find out if it's stdout or stderr, or if the URL format changed.
  for line in p.stderr:
    l = line.decode()
    print(f"[cloudflared stderr] {l.strip()}") # Print all stderr
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')

  for line in p.stdout:
    l = line.decode()
    print(f"[cloudflared stdout] {l.strip()}") # Print all stdout
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')

# Start the iframe_thread in the background
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

print("\n--- Starting ComfyUI server ---\n")

# Start ComfyUI in a non-blocking way using subprocess.Popen
# This allows the main script to continue running and keep daemon threads alive.
comfyui_process = subprocess.Popen(
    ["python", "main.py", "--dont-print-server"],
    cwd=os.getcwd(), # Ensure it runs in the ComfyUI directory
    stdout=subprocess.PIPE, # Capture stdout
    stderr=subprocess.PIPE # Capture stderr
)

# Function to monitor and print ComfyUI's output in a separate thread
def monitor_comfyui_output(process):
    for line in iter(process.stdout.readline, b''):
        print(f"[ComfyUI stdout] {line.decode().strip()}")
    for line in iter(process.stderr.readline, b''):
        print(f"[ComfyUI stderr] {line.decode().strip()}")

# Start monitoring in a separate daemon thread
threading.Thread(target=monitor_comfyui_output, args=(comfyui_process,), daemon=True).start()

# Keep the main thread alive indefinitely to support the daemon iframe_thread and comfyui_process.
# This loop will run until the cell is manually interrupted.
try:
    while True:
        time.sleep(1) # Keep the main thread active
        if comfyui_process.poll() is not None: # Check if ComfyUI process has terminated unexpectedly
            print("[ERROR] ComfyUI process terminated unexpectedly.")
            break # Exit the loop if ComfyUI stops
except KeyboardInterrupt:
    print("\nStopping ComfyUI and Cloudflared...")
finally:
    # Ensure ComfyUI process is terminated on cell stop
    if comfyui_process.poll() is None: # If ComfyUI is still running
        comfyui_process.terminate()
        comfyui_process.wait()
    print("Cleanup complete.")

/content/drive/MyDrive/ComfyUI

--- Installing ComfyUI requirements ---


--- Starting ComfyUI server ---

[ComfyUI stdout] [START] Security scan
[ComfyUI stdout] [DONE] Security scan
[ComfyUI stdout] ## ComfyUI-Manager: installing dependencies done.
[ComfyUI stdout] ** ComfyUI startup time: 2026-09-05 18:02:58.383
[ComfyUI stdout] ** Platform: Linux
[ComfyUI stdout] ** Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
[ComfyUI stdout] ** Python executable: /usr/bin/python3
[ComfyUI stdout] ** ComfyUI Path: /content/drive/MyDrive/ComfyUI
[ComfyUI stdout] ** ComfyUI Base Folder Path: /content/drive/MyDrive/ComfyUI
[ComfyUI stdout] ** User directory: /content/drive/MyDrive/ComfyUI/user
[ComfyUI stdout] ** ComfyUI-Manager config path: /content/drive/MyDrive/ComfyUI/user/__manager/config.ini
[ComfyUI stdout] ** Log path: /content/drive/MyDrive/ComfyUI/user/comfyui.log

ComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issue

### Run ComfyUI with localtunnel




In [2]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 1s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[INFO] [ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.13.15 environment at: /usr
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-09-05 17:56:54.880
** Platform: Linux
** Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/drive/MyDrive/ComfyUI
** ComfyUI Base Folder Path: /content/drive/MyDrive/ComfyUI
** User directory: /content/drive/MyDrive/Com

### Run ComfyUI with colab iframe (use only in case the previous way with localtunnel doesn't work)

You should see the ui appear in an iframe. If you get a 403 error, it's your firefox settings or an extension that's messing things up.

If you want to open it in another window use the link.

Note that some UI features like live image previews won't work because the colab iframe blocks websockets.

In [ ]:
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  from google.colab import output
  output.serve_kernel_port_as_iframe(port, height=1024)
  print("to open it in a window you can open this link here:")
  output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server
!cloudflared tunnel run --token eyJhIjoiOTYzY2I4OTYwNzJmYTRkY2IyZWEyMTMzYWM4NDQ1MmEiLCJ0IjoiOTE2ZGQxMTctNzgxZC00OTVjLTlkNDctMjE4NWZkMTQ4ZWUzIiwicyI6Ik1XTTROR0UzT0dJdFpUWTFZeTAwTWpNNExUazRZVFl0T0RjMFpXWmpPVFkyTXpZMyJ9

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[INFO] [ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.13.15 environment at: /usr
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-09-05 18:40:06.831
** Platform: Linux
** Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/drive/MyDrive/ComfyUI
** ComfyUI Base Folder Path: /content/drive/MyDrive/ComfyUI
** User directory: /content/drive/MyDrive/ComfyUI/user
** ComfyUI-Manager config path: /content/drive/MyDrive/ComfyUI/user/__manager/config.ini
** Log path

<IPython.core.display.Javascript object>

to open it in a window you can open this link here:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

FETCH ComfyRegistry Data [DONE]
[INFO] [ComfyUI-Manager] default cache updated: https://api.comfy.org/nodes
FETCH DATA from: https://raw.githubusercontent.com/ltdrdata/ComfyUI-Manager/main/custom-node-list.json [DONE]
[INFO] [ComfyUI-Manager] All startup tasks have been completed.
